In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 0 — SETUP & CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════
import os, zipfile, warnings, tempfile
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.3f}'.format)
pd.set_option('display.max_columns', 20)

BASE = os.getcwd()
ANA  = os.path.join(BASE, '08_Processed', 'analytical')
MRG  = os.path.join(BASE, '08_Processed', 'merged')

def save(df, name, folder=None):
    folder = folder or ANA
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, name)
    df.to_csv(path, index=False)
    print(f'  SAVED: {name}  |  {df.shape[0]:,} rows x {df.shape[1]} cols')
    return path

# OEWS hourly wages — BLS National Occupational Employment Statistics, May 2024
# Source: U.S. Bureau of Labor Statistics, https://www.bls.gov/oes/
WAGES_HR = {
    'w_housekeeping':  17.39,   # SOC 37-2012 Maids & Housekeeping Cleaners
    'w_childcare':     15.93,   # SOC 39-9011 Childcare Workers
    'w_cook':          18.14,   # SOC 35-2014 Cooks, Restaurant
    'w_eldercare':     22.64,   # SOC 21-1093 Social & Human Service Assistants
    'w_personal_care': 17.39,   # Mapped to housekeeping rate
}

# LGD baseline: 20% — standard Basel III residential first-lien mortgage
LGD = 0.20

# PSID state code -> U.S. state abbreviation
PSID_STATE = {
    1:'AL',  2:'AZ',  3:'AR',  4:'CA',  5:'CO',  6:'CT',  7:'DE',  8:'FL',
    9:'GA', 10:'ID', 11:'IL', 12:'IN', 13:'IA', 14:'KS', 15:'KY', 16:'LA',
   17:'ME', 18:'MD', 19:'MA', 20:'MI', 21:'MN', 22:'MS', 23:'MO', 24:'MT',
   25:'NE', 26:'NV', 27:'NH', 28:'NJ', 29:'NM', 30:'NY', 31:'NC', 32:'ND',
   33:'OH', 34:'OK', 35:'OR', 36:'PA', 37:'RI', 38:'SC', 39:'SD', 40:'TN',
   41:'TX', 42:'UT', 43:'VT', 44:'VA', 45:'WA', 46:'WV', 47:'WI', 48:'WY',
   49:'AK', 50:'HI', 51:'DC'
}

print(f'BASE: {BASE}')
print('Setup complete.')


BASE: C:\Users\Amirh\OneDrive - Wright State University\Research\DRMI_Research
Setup complete.


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — ACS STATE HOMEMAKER PREVALENCE RATES
# Memory-safe chunked reader for large ACS PUMS file
# ═══════════════════════════════════════════════════════════════════════
acs_zip = os.path.join(BASE, '06_ACS', 'raw', 'csv_pus.zip')
CHUNK = 100_000  # safe for any RAM size

# Detect available column names from header
with zipfile.ZipFile(acs_zip) as z:
    csv_files = sorted([f for f in z.namelist() if f.endswith('.csv')])
    sample = pd.read_csv(z.open(csv_files[0]), nrows=0)
    all_cols = list(sample.columns)

REQUIRED  = ['STATE', 'MAR', 'AGEP', 'SEX', 'ESR']
CHILD_TRY = ['NOC', 'NRC', 'OC', 'NCHILD', 'PAOC']
child_col = next((c for c in CHILD_TRY if c in all_cols), None)
USE_COLS  = REQUIRED + ([child_col] if child_col else [])
print(f'Reading {len(csv_files)} split files | columns: {USE_COLS}')

missing = [c for c in REQUIRED if c not in all_cols]
if missing:
    raise ValueError(f'Required ACS columns missing: {missing}')

# Process all split files in chunks — filter to married women 25-60
all_filtered = []
with zipfile.ZipFile(acs_zip) as z:
    for fname in csv_files:
        file_frames = []
        reader = pd.read_csv(z.open(fname), usecols=USE_COLS,
                              low_memory=False, chunksize=CHUNK)
        for chunk in reader:
            filtered = chunk[(chunk['MAR']==1) & chunk['AGEP'].between(25,60)].copy()
            if len(filtered) > 0:
                file_frames.append(filtered)
        if file_frames:
            df_f = pd.concat(file_frames, ignore_index=True)
            all_filtered.append(df_f)
            print(f'  {fname}: {df_f["STATE"].nunique()} states kept')

df_married = pd.concat(all_filtered, ignore_index=True)
print(f'Combined: {len(df_married):,} married adults | {df_married["STATE"].nunique()} states')

# Homemaker: female (SEX=2) + not in labor force (ESR=6)
# ACS ESR codes: 1=Employed, 2=Emp-not-at-work, 3=Unemployed,
#               4-5=Armed forces, 6=Not in labor force
df_married['IS_HM'] = ((df_married['SEX']==2) & (df_married['ESR']==6)).astype(float)
print(f'Overall homemaker rate: {100*df_married["IS_HM"].mean():.1f}%')

agg = {'N_MARRIED':('IS_HM','count'), 'N_HM':('IS_HM','sum'), 'HM_RATE':('IS_HM','mean')}
if child_col:
    agg['MEAN_CHILDREN'] = (child_col, 'mean')
state_hm = df_married.groupby('STATE').agg(**agg).reset_index()
state_hm.rename(columns={'STATE':'STATE_FIPS'}, inplace=True)
state_hm = state_hm[state_hm['N_MARRIED'] >= 50]

print(f'State homemaker rates: {len(state_hm)} states | Mean: {100*state_hm["HM_RATE"].mean():.1f}%')
save(state_hm, 'acs_state_homemaker_rates.csv', MRG)


Reading 4 split files | columns: ['STATE', 'MAR', 'AGEP', 'SEX', 'ESR', 'OC']
  psam_pusa.csv: 12 states kept
  psam_pusb.csv: 13 states kept
  psam_pusc.csv: 13 states kept
  psam_pusd.csv: 13 states kept
Combined: 4,089,399 married adults | 51 states
Overall homemaker rate: 13.1%
State homemaker rates: 51 states | Mean: 12.5%
  SAVED: acs_state_homemaker_rates.csv  |  51 rows x 5 cols


'C:\\Users\\Amirh\\OneDrive - Wright State University\\Research\\DRMI_Research\\08_Processed\\merged\\acs_state_homemaker_rates.csv'

In [11]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2 — IV DATASET CONSTRUCTION (PSID 2019 + NDCP Childcare Costs)
# FIX: drops pre-existing CC columns before merge to prevent _x/_y suffix bug
# ═══════════════════════════════════════════════════════════════════════
df_19 = pd.read_csv(os.path.join(ANA, 'drmi_study_pop_2019.csv'))
print(f'2019 study pop: {df_19.shape}')

# ── Drop any CC columns from prior runs (prevents _x/_y suffix collision) ─
cc_old = [c for c in df_19.columns
          if any(k in c.upper() for k in ('CC_WEEKLY','CC_ANNUAL','CC_COMPOSITE'))]
if cc_old:
    df_19 = df_19.drop(columns=cc_old)
    print(f'Dropped pre-existing CC columns: {cc_old}')

# ── Identify PSID state code column ───────────────────────────────────
state_col = 'STATE' if 'STATE' in df_19.columns else \
            next((c for c in df_19.columns if c.upper() in ('STATE_2019','STATE_CURRENT_2019')), None)
if state_col is None:
    state_col = [c for c in df_19.columns if 'STATE' in c.upper()][0]
print(f'State column: {state_col}')

df_19['STATE_ABBR'] = pd.to_numeric(df_19[state_col], errors='coerce').map(PSID_STATE)
n_mapped = df_19['STATE_ABBR'].notna().sum()
print(f'State mapping: {n_mapped:,}/{len(df_19):,} matched ({df_19["STATE_ABBR"].nunique()} states)')

# ── Load NDCP & build 2018 IV ─────────────────────────────────────────
df_iv = pd.read_csv(os.path.join(MRG, 'ndcp_iv_state_year.csv'))
print(f'NDCP: {df_iv.shape} | Years: {sorted(df_iv["NDCP_YEAR"].unique())}')

iv_2018 = df_iv[df_iv['NDCP_YEAR']==2018][['STATE_ABBR','CC_COMPOSITE']].copy()
iv_2018.rename(columns={'CC_COMPOSITE':'CC_WEEKLY_2018'}, inplace=True)
iv_2018['CC_ANNUAL_2018'] = iv_2018['CC_WEEKLY_2018'] * 52
iv_2018['LOG_CC_ANNUAL']  = np.log1p(iv_2018['CC_ANNUAL_2018'])

print(f'CC 2018 weekly: mean=${iv_2018["CC_WEEKLY_2018"].mean():.0f} '
      f'median=${iv_2018["CC_WEEKLY_2018"].median():.0f}')
print(f'CC 2018 annual: mean=${iv_2018["CC_ANNUAL_2018"].mean():,.0f} '
      f'median=${iv_2018["CC_ANNUAL_2018"].median():,.0f}')

# ── Merge ──────────────────────────────────────────────────────────────
# Both keys are already strings from PSID_STATE map / NDCP csv — no need for extra conversion
df_19_iv = df_19.merge(iv_2018, on='STATE_ABBR', how='left')

matched = df_19_iv['CC_ANNUAL_2018'].notna().sum()
print(f'IV match rate: {matched:,}/{len(df_19_iv):,} ({100*matched/len(df_19_iv):.1f}%)')

# ── First-stage direction check ────────────────────────────────────────
state_check = df_19_iv.groupby('STATE_ABBR').agg(
    HM_RATE=('IS_HOMEMAKER','mean'),
    CC_ANNUAL=('CC_ANNUAL_2018','mean')
).dropna()
corr = state_check['HM_RATE'].corr(state_check['CC_ANNUAL'])
print(f'State-level corr(HM_rate, CC_annual) = {corr:.4f}  (n={len(state_check)} states)')
print(f'Individual-level first-stage F-stat will be the definitive validity test.')

save(df_19_iv, 'psid_2019_with_iv.csv')


2019 study pop: (2116, 88)
Dropped pre-existing CC columns: ['CC_WEEKLY_2018', 'CC_ANNUAL_2018']
State column: STATE_2019
State mapping: 2,109/2,116 matched (50 states)
NDCP: (561, 6) | Years: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]
CC 2018 weekly: mean=$174 median=$163
CC 2018 annual: mean=$9,058 median=$8,495
IV match rate: 1,670/2,116 (78.9%)
State-level corr(HM_rate, CC_annual) = -0.0616  (n=41 states)
Individual-level first-stage F-stat will be the definitive validity test.
  SAVED: psid_2019_with_iv.csv  |  2,116 rows x 89 cols


'C:\\Users\\Amirh\\OneDrive - Wright State University\\Research\\DRMI_Research\\08_Processed\\analytical\\psid_2019_with_iv.csv'

In [13]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — DSV HARMONIZATION
# Creates comparable DSV variables across both PSID waves
# DSV = Domestic Service Value = Σ(weekly_hours_i × hourly_wage_i) × 52
# ═══════════════════════════════════════════════════════════════════════
df_19 = pd.read_csv(os.path.join(ANA, 'drmi_study_pop_2019.csv'))
df_21 = pd.read_csv(os.path.join(ANA, 'drmi_study_pop_2021.csv'))

w_hk = WAGES_HR['w_housekeeping']   # $17.39/hr  SOC 37-2012
w_cc = WAGES_HR['w_childcare']      # $15.93/hr  SOC 39-9011
w_ec = WAGES_HR['w_eldercare']      # $22.64/hr  SOC 21-1093
w_ck = WAGES_HR['w_cook']           # $18.14/hr  SOC 35-2014
w_pc = WAGES_HR['w_personal_care']  # $17.39/hr  mapped to housekeeping

def build_dsv(df):
    df = df.copy()
    # Clean: recode PSID missing (999) to NaN, cap at 168 h/wk physical maximum
    for col in df.columns:
        if 'WIFE_HRS' in col:
            df[col] = pd.to_numeric(df[col], errors='coerce').replace(999, np.nan).clip(0, 168)
    def get(c): return df[c].fillna(0) if c in df.columns else pd.Series(0, index=df.index)

    # DSV_COMMON: housework + childcare + eldercare (available both waves)
    df['DSV_COMMON'] = (
        get('WIFE_HRS_HOUSEWORK') * w_hk +
        get('WIFE_HRS_CHILDCARE') * w_cc +
        get('WIFE_HRS_ADULT_CARE') * w_ec
    ) * 52

    # DSV_FULL: adds shopping + personal care (2019 only)
    df['DSV_FULL'] = (
        get('WIFE_HRS_HOUSEWORK')    * w_hk +
        get('WIFE_HRS_CHILDCARE')    * w_cc +
        get('WIFE_HRS_ADULT_CARE')   * w_ec +
        get('WIFE_HRS_SHOPPING')     * w_ck +
        get('WIFE_HRS_PERSONAL_CARE')* w_pc
    ) * 52

    # DRMI = DSV / (Mortgage_Balance × LGD_baseline)
    ead = df['MORTGAGE_BALANCE'].fillna(0).where(df['MORTGAGE_BALANCE'] > 0)
    df['DRMI_COMMON'] = np.where(
        ead.notna() & (df['DSV_COMMON'] > 0),
        df['DSV_COMMON'] / (ead * LGD), np.nan
    )
    return df

df_19 = build_dsv(df_19)
df_21 = build_dsv(df_21)

# Verify comparability
print('DSV_COMMON (3 vars, both waves) — Homemakers only:')
for yr, df in [('2019', df_19), ('2021', df_21)]:
    hm = df[df['IS_HOMEMAKER']==1]
    nz = hm[hm['DSV_COMMON']>0]['DSV_COMMON']
    print(f'  [{yr}] n={len(nz):,}  mean=${nz.mean():,.0f}  median=${nz.median():,.0f}')

print('Use DSV_COMMON for panel regressions. Use DSV_FULL for 2019 cross-section only.')
save(df_19, 'drmi_study_pop_2019.csv')
save(df_21, 'drmi_study_pop_2021.csv')


DSV_COMMON (3 vars, both waves) — Homemakers only:
  [2019] n=313  mean=$75,527  median=$56,501
  [2021] n=122  mean=$56,117  median=$45,504
Use DSV_COMMON for panel regressions. Use DSV_FULL for 2019 cross-section only.
  SAVED: drmi_study_pop_2019.csv  |  2,116 rows x 88 cols
  SAVED: drmi_study_pop_2021.csv  |  1,179 rows x 58 cols


'C:\\Users\\Amirh\\OneDrive - Wright State University\\Research\\DRMI_Research\\08_Processed\\analytical\\drmi_study_pop_2021.csv'

In [15]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — WINSORIZATION & LOG TRANSFORMS
# ═══════════════════════════════════════════════════════════════════════
df_19 = pd.read_csv(os.path.join(ANA, 'drmi_study_pop_2019.csv'))
df_21 = pd.read_csv(os.path.join(ANA, 'drmi_study_pop_2021.csv'))

WINSOR_VARS = [
    'DRMI_COMMON', 'DSV_COMMON', 'DSV_FULL',
    'MORTGAGE_BALANCE', 'HOME_VALUE', 'LTV', 'DTI_PROXY',
    'WIFE_HRS_HOUSEWORK', 'WIFE_HRS_CHILDCARE', 'WIFE_HRS_ADULT_CARE',
    'WIFE_WAGES_SALARY_RAW',
]
LOG_VARS = ['DRMI_COMMON', 'DSV_COMMON', 'MORTGAGE_BALANCE', 'HOME_VALUE', 'WIFE_WAGES_SALARY_RAW']

def winsorize_df(df, year):
    df = df.copy()
    for col in WINSOR_VARS:
        if col not in df.columns: continue
        s = pd.to_numeric(df[col], errors='coerce')
        if s.dropna().std() == 0: continue
        p1, p99 = s.quantile(0.01), s.quantile(0.99)
        df[f'{col}_W'] = s.clip(lower=p1, upper=p99)
    for col in LOG_VARS:
        src = f'{col}_W' if f'{col}_W' in df.columns else col
        if src in df.columns:
            df[f'LOG_{col}'] = np.log1p(pd.to_numeric(df[src], errors='coerce').clip(lower=0))
    return df

df_19 = winsorize_df(df_19, '2019')
df_21 = winsorize_df(df_21, '2021')

# Verify DRMI_COMMON_W is clean
for yr, df in [('2019',df_19),('2021',df_21)]:
    hm = df[df['IS_HOMEMAKER']==1]
    if 'DRMI_COMMON_W' in df.columns:
        s = hm['DRMI_COMMON_W'].dropna()
        print(f'[{yr}] DRMI_COMMON_W HM: mean={s.mean():.3f} p99={s.quantile(0.99):.3f} skew={s.skew():.2f}')
    # Check log transform is finite
    if 'LOG_DRMI_COMMON' in df.columns:
        n_inf = np.isinf(df['LOG_DRMI_COMMON']).sum()
        print(f'  LOG_DRMI_COMMON: {n_inf} inf values (should be 0)')

save(df_19, 'drmi_study_pop_2019.csv')
save(df_21, 'drmi_study_pop_2021.csv')


[2019] DRMI_COMMON_W HM: mean=4.550 p99=44.178 skew=3.80
  LOG_DRMI_COMMON: 0 inf values (should be 0)
[2021] DRMI_COMMON_W HM: mean=2.589 p99=20.955 skew=3.27
  LOG_DRMI_COMMON: 0 inf values (should be 0)
  SAVED: drmi_study_pop_2019.csv  |  2,116 rows x 88 cols
  SAVED: drmi_study_pop_2021.csv  |  1,179 rows x 58 cols


'C:\\Users\\Amirh\\OneDrive - Wright State University\\Research\\DRMI_Research\\08_Processed\\analytical\\drmi_study_pop_2021.csv'

In [17]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — DTI COVERAGE ENHANCEMENT
# DTI_PROXY_V2 = monthly_mortgage_payment / monthly_income
# ═══════════════════════════════════════════════════════════════════════
df_19 = pd.read_csv(os.path.join(ANA, 'drmi_study_pop_2019.csv'))
df_21 = pd.read_csv(os.path.join(ANA, 'drmi_study_pop_2021.csv'))

def build_dti(df, year):
    df = df.copy()
    df['BEST_INCOME'] = np.nan
    # Priority: family income first, then head labor, then wife wages
    for col in ['FAMILY_INCOME_2018','FAMILY_INCOME_2020','FAMILY_INCOME',
                'HEAD_LABOR_INC_2018','HEAD_LABOR_INC','WIFE_WAGES_SALARY_RAW']:
        if col in df.columns:
            s = pd.to_numeric(df[col], errors='coerce')
            df['BEST_INCOME'] = df['BEST_INCOME'].fillna(s.where(s > 500))
    n = df['BEST_INCOME'].notna().sum()
    print(f'  [{year}] BEST_INCOME: {n:,}/{len(df):,} ({100*n/len(df):.1f}%)')

    df['DTI_PROXY_V2'] = np.where(
        df['BEST_INCOME'].notna() & (df['BEST_INCOME']>0) & df['MORTGAGE_PAYMENT_MO'].notna(),
        df['MORTGAGE_PAYMENT_MO'] / (df['BEST_INCOME']/12),
        np.nan
    )
    df['DTI_PROXY_V2'] = df['DTI_PROXY_V2'].clip(upper=2.0)
    n_dti = df['DTI_PROXY_V2'].notna().sum()
    print(f'  [{year}] DTI_PROXY_V2: {n_dti:,}/{len(df):,} ({100*n_dti/len(df):.1f}%)')
    if n_dti > 0:
        print(f'  mean={df["DTI_PROXY_V2"].mean():.3f} median={df["DTI_PROXY_V2"].median():.3f}')
    return df

df_19 = build_dti(df_19, '2019')
df_21 = build_dti(df_21, '2021')

save(df_19, 'drmi_study_pop_2019.csv')
save(df_21, 'drmi_study_pop_2021.csv')


  [2019] BEST_INCOME: 1,548/2,116 (73.2%)
  [2019] DTI_PROXY_V2: 1,548/2,116 (73.2%)
  mean=0.531 median=0.329
  [2021] BEST_INCOME: 1,169/1,179 (99.2%)
  [2021] DTI_PROXY_V2: 1,169/1,179 (99.2%)
  mean=0.184 median=0.137
  SAVED: drmi_study_pop_2019.csv  |  2,116 rows x 88 cols
  SAVED: drmi_study_pop_2021.csv  |  1,179 rows x 58 cols


'C:\\Users\\Amirh\\OneDrive - Wright State University\\Research\\DRMI_Research\\08_Processed\\analytical\\drmi_study_pop_2021.csv'

In [19]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — MASTER ANALYTICAL DATASET ASSEMBLY
# Produces: 08_Processed/analytical/DRMI_MASTER_DATASET.csv
# ═══════════════════════════════════════════════════════════════════════
df_19 = pd.read_csv(os.path.join(ANA, 'drmi_study_pop_2019.csv'))
df_21 = pd.read_csv(os.path.join(ANA, 'drmi_study_pop_2021.csv'))

# Add STATE_ABBR to 2021
if 'STATE_ABBR' not in df_21.columns and 'STATE' in df_21.columns:
    df_21['STATE_ABBR'] = df_21['STATE'].map(PSID_STATE)

# Merge 2019 with IV instrument
iv_path = os.path.join(ANA, 'psid_2019_with_iv.csv')
if os.path.isfile(iv_path):
    df_iv  = pd.read_csv(iv_path)
    fid_19 = next((c for c in ('FAMILY_ID_2019','FAMILY_ID','ER72002') if c in df_19.columns), None)
    fid_iv = next((c for c in ('FAMILY_ID_2019','FAMILY_ID','ER72002') if c in df_iv.columns), None)
    iv_cols= [c for c in ('CC_WEEKLY_2018','CC_ANNUAL_2018','STATE_ABBR')
              if c in df_iv.columns and c not in df_19.columns]
    if fid_19 and fid_iv and iv_cols:
        df_19 = df_19.merge(
            df_iv[[fid_iv]+iv_cols].drop_duplicates(fid_iv),
            left_on=fid_19, right_on=fid_iv, how='left'
        )
        n_iv = df_19['CC_ANNUAL_2018'].notna().sum() if 'CC_ANNUAL_2018' in df_19.columns else 0
        print(f'IV merged: {n_iv:,}/{len(df_19):,} matched')

# Standardize column names across years
def std_cols(df, year):
    df = df.copy()
    df['YEAR'] = year
    for old, new in [('FAMILY_ID_2019','FAMILY_ID'),('FAMILY_ID_2021','FAMILY_ID'),
                     ('STATE_2019','STATE'),('STATE_2021','STATE'),
                     ('FAMILY_INCOME_2018','FAMILY_INCOME'),('FAMILY_INCOME_2020','FAMILY_INCOME')]:
        if old in df.columns and new not in df.columns:
            df.rename(columns={old:new}, inplace=True)
    return df

df_19 = std_cols(df_19, 2019)
df_21 = std_cols(df_21, 2021)

# Add IV columns as NaN for 2021 (IV only covers 2018)
iv_extra = [c for c in ('CC_WEEKLY_2018','CC_ANNUAL_2018','LOG_CC_ANNUAL')
            if c in df_19.columns and c not in df_21.columns]
for c in iv_extra:
    df_21[c] = np.nan

common = sorted(set(df_19.columns) & set(df_21.columns))
df_master = pd.concat([df_19[common], df_21[common]], ignore_index=True)
print(f'Master: {df_master.shape}  ({len(df_master[df_master.YEAR==2019]):,} in 2019, {len(df_master[df_master.YEAR==2021]):,} in 2021)')

hm = df_master[df_master['IS_HOMEMAKER']==1]
di = df_master[df_master['IS_HOMEMAKER']==0]
print(f'Treatment: {len(hm):,} homemakers | Control: {len(di):,} dual-income')
print(f'Defaults: {int(df_master["DEFAULT"].sum())} ({100*df_master["DEFAULT"].mean():.2f}%)')

save(df_master, 'DRMI_MASTER_DATASET.csv')


Master: (3295, 60)  (2,116 in 2019, 1,179 in 2021)
Treatment: 439 homemakers | Control: 2,856 dual-income
Defaults: 55 (1.67%)
  SAVED: DRMI_MASTER_DATASET.csv  |  3,295 rows x 60 cols


'C:\\Users\\Amirh\\OneDrive - Wright State University\\Research\\DRMI_Research\\08_Processed\\analytical\\DRMI_MASTER_DATASET.csv'

In [23]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 7 — DATA QUALITY & PUBLICATION READINESS CHECK
# ═══════════════════════════════════════════════════════════════════════
master = pd.read_csv(os.path.join(ANA, 'DRMI_MASTER_DATASET.csv'))
hm = master[master['IS_HOMEMAKER']==1]
di = master[master['IS_HOMEMAKER']==0]

print(f'Master: {master.shape[0]:,} obs x {master.shape[1]} cols')
print(f'  2019: {len(master[master.YEAR==2019]):,} | 2021: {len(master[master.YEAR==2021]):,}')
print(f'  Homemakers: {len(hm):,} ({100*len(hm)/len(master):.1f}%) | Control: {len(di):,}')
print(f'  Defaults: {int(master["DEFAULT"].sum())} ({100*master["DEFAULT"].mean():.2f}%)')

drmi = next((c for c in ('DRMI_COMMON_W','DRMI_COMMON') if c in master.columns), None)
dsv  = next((c for c in ('DSV_COMMON_W','DSV_COMMON') if c in master.columns), None)

print(f'\nDRMI gap: HM={hm[drmi].mean():.3f} vs DI={di[drmi].mean():.3f} '
      f'ratio={hm[drmi].mean()/di[drmi].mean():.2f}x')
print(f'DSV gap:  HM=${hm[dsv].mean():,.0f} vs DI=${di[dsv].mean():,.0f} '
      f'diff=${hm[dsv].mean()-di[dsv].mean():,.0f}')
print(f'DRMI skewness: {hm[drmi].dropna().skew():.2f} (log transform recommended)')

print('\nVARIABLE COVERAGE:')
for col, label in [('IS_HOMEMAKER','Treatment'), ('DEFAULT','Outcome'),
                    (drmi,'DRMI (winsorized)'), (dsv,'DSV (winsorized)'),
                    ('LTV_W','LTV'), ('DTI_PROXY_V2','DTI'),
                    ('N_CHILDREN','N Children'), ('AGE_HEAD','Age'),
                    ('STATE_ABBR','State FE'), ('CC_ANNUAL_2018','IV')]:
    if col and col in master.columns:
        pct = 100*master[col].notna().mean()
        flag = '' if pct>=70 else ' <-- LOW'
        print(f'  {label:<22} {pct:>6.1f}% non-null{flag}')

print('\nSANITY CHECKS:')
checks = [
    ('No duplicate rows', master.duplicated().sum()==0),
    ('IS_HOMEMAKER is 0/1', set(master['IS_HOMEMAKER'].unique()).issubset({0,1})),
    ('DEFAULT is 0/1', set(master['DEFAULT'].dropna().unique()).issubset({0.0,1.0})),
    ('LTV range 0-2', master['LTV_W'].between(0,2).all()),
    ('No log transform infinities',
     all(not np.isinf(master[c]).any() for c in master.columns if 'LOG_' in c)),
    ('DRMI HM > DI', hm[drmi].mean() > di[drmi].mean()),
]
for label, result in checks:
    print(f'  [{"PASS" if result else "FAIL"}] {label}')


Master: 3,295 obs x 60 cols
  2019: 2,116 | 2021: 1,179
  Homemakers: 439 (13.3%) | Control: 2,856
  Defaults: 55 (1.67%)

DRMI gap: HM=4.000 vs DI=1.938 ratio=2.06x
DSV gap:  HM=$66,844 vs DI=$26,617 diff=$40,227
DRMI skewness: 4.10 (log transform recommended)

VARIABLE COVERAGE:
  Treatment               100.0% non-null
  Outcome                 100.0% non-null
  DRMI (winsorized)        89.8% non-null
  DSV (winsorized)        100.0% non-null
  LTV                     100.0% non-null
  DTI                      82.5% non-null
  N Children              100.0% non-null
  Age                     100.0% non-null
  IV                       50.7% non-null <-- LOW

SANITY CHECKS:
  [PASS] No duplicate rows
  [PASS] IS_HOMEMAKER is 0/1
  [PASS] DEFAULT is 0/1
  [PASS] LTV range 0-2
  [PASS] No log transform infinities
  [PASS] DRMI HM > DI
